In [ ]:
import os
import json
import pandas as pd

os.chdir("/Users/astitva/Project Basant Panchami")
file_path = "data/raw/t20s_male_json/211028.json"
with open(file_path) as f:
  match=json.load(f)
match.keys()
print (match)

In [2]:
len(match["innings"])

2

In [3]:
match["innings"][0]["overs"][0].keys()


dict_keys(['over', 'deliveries'])

In [30]:
rows = []

for innings_index, innings in enumerate(match["innings"], start=1):
    for over in innings["overs"]:
        over_number = over["over"]

        for delivery in over["deliveries"]:
            runs_bat = delivery["runs"]["batter"]
            runs_extra = delivery["runs"]["extras"]
            total_runs = delivery["runs"]["total"]

            is_wicket = 1 if "wickets" in delivery else 0

            # extras type handling (CRITICAL)
            extras_type = delivery.get("extras", {})
            is_wide = 1 if "wides" in extras_type else 0
            is_noball = 1 if "noballs" in extras_type else 0

            # dot ball = legal delivery with 0 total runs
            is_dot = 1 if (total_runs == 0 and is_wide == 0) else 0

            rows.append({
                "innings": innings_index,
                "over": over_number,
                "runs_off_bat": runs_bat,
                "extras": runs_extra,
                "total_runs": total_runs,
                "is_wicket": is_wicket,
                "is_wide": is_wide,
                "is_dot": is_dot
            })

ball_df= pd.DataFrame(rows)
ball_df.head()



,innings,over,runs_off_bat,extras,total_runs,is_wicket,is_wide,is_dot
0,1,0,0,0,0,0,0,1
1,1,0,1,0,1,0,0,0
2,1,0,0,0,0,0,0,1
3,1,0,0,0,0,0,0,1
4,1,0,0,0,0,0,0,1


In [32]:
ball_df.groupby(["innings", "over"]).agg(
    runs_per_over=("total_runs", "sum"),
    wickets=("is_wicket", "sum")
).reset_index().head()


,innings,over,runs_per_over,wickets
0,1,0,4,0
1,1,1,2,0
2,1,2,14,0
3,1,3,9,1
4,1,4,14,0


In [ ]:
ball_df["dot_ball"]=(df["runs"]==0).astype(int)
ball_df.head()

,innings,over,runs,wicket,dot_ball
0,1,0,0,0,1
1,1,0,1,0,0
2,1,0,0,0,1
3,1,0,0,0,1
4,1,0,0,0,1


In [ ]:
over_df = ball_df.groupby(["innings", "over"]).agg(
    runs_per_over=("runs", "sum"),
    dot_balls=("dot_ball", "sum"),
    wickets=("wicket", "sum")
).reset_index()

over_df.head()


,innings,over,runs_per_over,dot_balls,wickets
0,1,0,4,4,0
1,1,1,2,5,0
2,1,2,14,1,0
3,1,3,9,3,1
4,1,4,14,0,0


In [ ]:
ball_df.columns



Index(['innings', 'over', 'runs', 'wicket', 'dot_ball'], dtype='object')

In [34]:
def phase(over):
    if over <= 6:
        return 'powerplay'
    elif over <= 15:
        return 'middle'
    else:
        return 'death'

over_df['phase'] = over_df['over'].apply(phase)


In [35]:
over_df['momentum'] = (
    over_df
    .groupby('innings')['runs_per_over']
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)



In [ ]:
over_df['pressure'] = (
    0.6 * (over_df['dot_balls'] / 6) +
    0.4 * over_df['wickets'].clip(0, 1)
)


In [ ]:
def phase_adjusted_pressure(row):
    dots = row['dot_balls'] / 6
    wkts = min(row['wickets'], 1)

    if row['phase'] == 'powerplay':
        return 0.7 * dots + 0.3 * wkts

    elif row['phase'] == 'middle':
        return 0.4 * dots + 0.6 * wkts

    else:  # death
        return 0.8 * dots + 0.2 * wkts
over_df['phase_pressure'] = over_df.apply(phase_adjusted_pressure, axis=1)


In [ ]:
over_df['aggression_index'] = (
    over_df['runs_per_over'] /
    (1 + over_df['pressure'])
)


In [36]:
over_df_basic = over_df.copy()

over_df_enriched = (
    ball_df
    .groupby(["innings", "over"])
    .agg(
        runs_per_over=("total_runs", "sum"),
        intent_runs=("runs_off_bat", "sum"),
        dot_balls=("is_dot", "sum"),
        wides=("is_wide", "sum"),
        wickets=("is_wicket", "sum")
    )
    .reset_index()
)

over_df_enriched.head()



,innings,over,runs_per_over,intent_runs,dot_balls,wides,wickets
0,1,0,4,3,4,0,0
1,1,1,2,1,5,1,0
2,1,2,14,14,1,0,0
3,1,3,9,9,3,0,1
4,1,4,14,11,0,1,0


In [39]:
over_df = over_df_enriched.merge(
    over_df_basic[['innings','over','pressure','momentum','phase']],
    on=['innings','over'],
    how='left'
)
over_df.head()



,innings,over,runs_per_over,intent_runs,dot_balls,wides,wickets,pressure,momentum,phase
0,1,0,4,3,4,0,0,0.4,4.000000,powerplay
1,1,1,2,1,5,1,0,0.5,3.000000,powerplay
2,1,2,14,14,1,0,0,0.1,6.666667,powerplay
3,1,3,9,9,3,0,1,0.7,8.333333,powerplay
4,1,4,14,11,0,1,0,0.0,12.333333,powerplay


In [40]:
over_df['legal_balls'] = 6 - over_df['wides']

over_df['ERR'] = (
    over_df['intent_runs'] /
    (over_df['legal_balls'] - over_df['dot_balls']).clip(lower=1)
)

phase_max = (
    over_df
    .groupby('phase')['runs_per_over']
    .quantile(0.9)
)

over_df['phase_max_runs'] = over_df['phase'].map(phase_max)

over_df['PNP'] = (
    over_df['runs_per_over'] /
    over_df['phase_max_runs']
)

over_df['SI'] = (
    0.6 * over_df['ERR'] +
    0.4 * over_df['PNP']
)
over_df[['over','intent_runs','dot_balls','wides',
         'ERR','PNP','SI']].head(10)





,over,intent_runs,dot_balls,wides,ERR,PNP,SI
0,0,3,4,0,1.500000,0.312500,1.025000
1,1,1,5,1,1.000000,0.156250,0.662500
2,2,14,1,0,2.800000,1.093750,2.117500
3,3,9,3,0,3.000000,0.703125,2.081250
4,4,11,0,1,2.200000,1.093750,1.757500
5,5,7,2,0,1.750000,0.546875,1.268750
6,6,8,2,0,2.000000,0.625000,1.450000
7,7,10,0,0,1.666667,0.847458,1.338983
8,8,14,0,0,2.333333,1.186441,1.874576
9,9,11,0,0,1.833333,0.932203,1.472881


In [42]:
over_df['risk_indicator'] = (
    (over_df['dot_balls'] / over_df['legal_balls']) +
    over_df['wickets']
)

over_df['pressure_amp'] = 1 + over_df['pressure']

over_df['aggression_index'] = (
    over_df['SI'] *
    over_df['pressure_amp'] *
    (1 + over_df['risk_indicator'])
)

over_df[['over','SI','pressure','risk_indicator','aggression_index']].head(10)


,over,SI,pressure,risk_indicator,aggression_index
0,0,1.025000,0.4,0.666667,2.391667
1,1,0.662500,0.5,1.000000,1.987500
2,2,2.117500,0.1,0.166667,2.717458
3,3,2.081250,0.7,1.500000,8.845312
4,4,1.757500,0.0,0.000000,1.757500
5,5,1.268750,0.6,1.333333,4.736667
6,6,1.450000,0.2,0.333333,2.320000
7,7,1.338983,0.0,0.000000,1.338983
8,8,1.874576,0.0,0.000000,1.874576
9,9,1.472881,0.0,0.000000,1.472881
